# Modeling example
The following notebook is a detailed modeling example that goes into further detail on how to analyze returns and use their autocorrelation to build a trading strategy.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

notebook_file = globals().get("__vsc_ipynb_file__")
NOTEBOOK_DIR = Path(notebook_file).resolve().parent if notebook_file else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))

from evaluation import calc_key_figures, plot_key_figures

DATE = "AsOfDate"

## Load prices and plot

In [ ]:
prices = pl.read_csv(NOTEBOOK_DIR / "example_prices.csv", try_parse_dates=True).rename({"dates": DATE})
instruments = [c for c in prices.columns if c != DATE]

fig, ax = plt.subplots()
for c in instruments:
    ax.plot(prices[DATE].to_numpy(), prices[c].to_numpy(), label=c)
ax.legend()
ax.grid()

## Compute returns and normalized returns, using a rolling volatility (standard deviation) estimate
Normalization is typically done before analysis, in order to get the different assets on the same scale, and simoultanously reduce the heteroscedasticity (volatility clusters) in the time series.

#### Instrument returns
As instrument returns we simply use the price differences between consecutive days; $r_t = p_t-p_{t-1}$ for prices $p\in \mathbb{R}^{T\times M}$.
For analysis, one could also use e.g logarithmic returns, but your positions will be evaluated on price differences.

In [ ]:
ret = prices.select(DATE, pl.col(instruments).diff())

#### Volatility estimates
We usually want a rolling estimate of volatility, that can adapt when the risks are changing and therefore reduce heteroscedasticity in the data. Using a sliding window of $N$ days, one can estimate/predict the volatility at day $t+1$ as $\hat{\sigma}_{t+1} = \sqrt{\frac{1}{N} \sum_{k=t-N}^{k=t}{r_t^2}}$. 

Notice that we don't adjust for the mean in this estimate, which is typically done when estimating standard deviation. We skip this since we assume that $E[r_t] \approx 0$ for daily returns, and the average on a short time frame might be a worse estimate of the mean than simply using zero. There are many other ways to estimate volatility, for example using an ARCH or GARCH model.

In this case we are using a sliding window of 100 days, which is easy to compute using pandas. Notice that the estimates are shifted 1 day since the rolling window up to day $t$ is used to predict the volatility att day $t+1$.

Positions can be scaled by the volatility estimates in order to achieve the normalized returns when trading.

In [ ]:
vol = ret.select(DATE, pl.col(instruments).pow(2).rolling_mean(100, min_samples=10).sqrt().shift(1))
norm_ret = ret.select(DATE, *[(pl.col(c) / vol[c]).fill_nan(None).alias(c) for c in instruments])

### Study the effect of normalization
By estimating the volatility over the whole time series, we see that all normalized return series now have about unit volatility, although it seems like we slightly tend to underestimate, since all instruments got realized volatilities greater than one.

In [ ]:
print(norm_ret.select(pl.col(instruments).std()))


#### Plot the normalized returns in the same figure as the raw returns, here for a single instrument. 
We see that we manage to adjust for heteroscedasticity, i.e that the volatility seem more constant over time.

In [ ]:
instrument = instruments[0]
fig, ax = plt.subplots()
ax.plot(ret[DATE].to_numpy(), ret[instrument].to_numpy(), label="Raw returns")
ax.plot(norm_ret[DATE].to_numpy(), norm_ret[instrument].to_numpy(), label="Normalized returns")
ax.legend()
ax.grid()

## Research example: Estimate autocorrelations for varied block sizes of accumulated returns
In order to find any pattern between past and future returns, on various time horizons, we can estimate the correlation between lagged returns accumulated into consecutive blocks, i.e: $corr(b_{n-1},b_n)$, where $b = \big\{\sum_{t=0}^K{r_t}, \sum_{t=K}^{2K}{r_t}, ...,  \sum_{t=T-K}^{T}{r_t}\big\}$, where $K$ refers to the block size.

In [ ]:
def calc_block_ac(ret: pl.DataFrame, max_block_size: int) -> pl.DataFrame:
    # Estimates correlation between consecutive blocks of returns (autocorrelation at lag 1), with block sizes 1, ..., max_block_size.
    # We make use of the fact that blocks of N days can be constructed in N different ways (shifted to each other)
    # to achieve the maximum sample size.
    cols = [c for c in ret.columns if c != DATE]
    cumulative = np.cumsum(ret.select(cols).drop_nulls().to_numpy(), axis=0)

    def offset_correlation(size: int, offset: int) -> np.ndarray:
        # loop over all possible shifts (ways of constructing the blocks)
        indices = np.arange(0, len(cumulative), size) - offset
        # Construct blocks of n_days return (difference of cumulative sum of returns with n_days sampling frequency)
        blocks = np.diff(cumulative[indices[indices >= 0]], axis=0)
        # Correlation between consecutive blocks
        previous = blocks[:-1] - blocks[:-1].mean(axis=0)
        following = blocks[1:] - blocks[1:].mean(axis=0)
        return (previous * following).sum(axis=0) / np.sqrt(
            (previous**2).sum(axis=0) * (following**2).sum(axis=0)
        )

    # loop over all block sizes
    rows = [
        # Mean over all different shifts
        np.mean([offset_correlation(size, offset) for offset in range(1, size + 1)], axis=0)
        for size in range(1, max_block_size + 1)
    ]
    return pl.DataFrame(np.asarray(rows), schema=cols).insert_column(
        0, pl.Series("block_size", range(1, max_block_size + 1))
    )

In [ ]:
max_block = 50
autocorrs = calc_block_ac(norm_ret, max_block)

### Plot the estimated autocorrelations
The autocorrelations seem positive for larger block sizes, but is it significant?

In [ ]:
mean_ac = autocorrs.select(pl.mean_horizontal(pl.col(instruments))).to_series()
fig, ax = plt.subplots()
ax.plot(autocorrs["block_size"].to_numpy(), mean_ac.to_numpy(), color="r", linewidth=2)
ax.set_ylabel("Autocorrelation")
ax.set_xlabel("Block size")
ax.grid()

### Estimate standard deviation for autocorrelation estimates on shuffled returns (true autocorrelation=0)
Takes about 30s to compute. Use this to test the significance of the autocorrelation estimates on returns with correct order.

In [ ]:
def shuffled_autocorrelations(
    ret: pl.DataFrame, max_block_size: int, samples: int, *, rng: np.random.Generator
) -> np.ndarray:
    # Construct dataframe of max_block samples of shuffled returns
    return np.column_stack([
        # Compute auto correlation estimates for shuffled returns (true autocorrelation is zero, since shuffled)
        calc_block_ac(
            ret.sample(fraction=1.0, shuffle=True, seed=int(rng.integers(2**32))), max_block_size
        ).select(pl.mean_horizontal(pl.exclude("block_size"))).to_series().to_numpy()
        for _ in range(samples)
    ])


ac_shuffled = shuffled_autocorrelations(norm_ret, max_block, 50, rng=np.random.default_rng(1))
# Standard deviations for the estimates, for each block size
std_shuffled = ac_shuffled.std(axis=1, ddof=1)

### Plot the estimated autocorrelations and $\pm 2$ standard deviations of the estimates on shuffled returns, i.e ~5% confidence interval

In [ ]:
fig, ax = plt.subplots()
ax.fill_between(autocorrs["block_size"].to_numpy(), 2 * std_shuffled, -2 * std_shuffled,
                label="+-2 stdev for zero autocorrelation")
ax.plot(autocorrs["block_size"].to_numpy(), mean_ac.to_numpy(), color="r", label="Mean autocorrelation for returns")
ax.set_ylabel("Autocorrelation")
ax.set_xlabel("Block size")
ax.legend()
ax.grid()

### Comment
The autocorrelation estimates seem significant for block sizes larger than ~20. This indicates that it should be a good idea to follow trends, i.e to take a positive position (go long) when the instrument returns have been positive on a period of sufficient length, and vise versa go short when the returns have been negative.

### Other hints for studying the data
What do the return distributions look like? What about the correlations between different assets?

## Trend model
Use the sign of returns accumulated on block sizes of 50 days as trading signal, and risk adjust by volatility estimates (as in the normalization above). 
Make sure that you only use information up to time $t-1$ for determining the position at time $t$! This is very important since the different markets close at different hours, and otherwise you might have forward looking bias, i.e using future information that was not yet available when determining the positions historically.

In [ ]:
def trend_model(ret: pl.DataFrame, trend_window: int = 50, vol_window: int = 100) -> pl.DataFrame:
    cols = [c for c in ret.columns if c != DATE]
    values = ret.select(cols).to_numpy()
    positions = np.full_like(values, np.nan)
    # loop over all dates
    for t in range(max(trend_window, vol_window), ret.height):
        # Volatility estimate; standard deviation on the last vol_window days, up to t
        vol = np.sqrt(np.nanmean(values[t - vol_window:t + 1] ** 2, axis=0))
        # Mean return between t-trend_window and t
        block_ret = np.nansum(values[t - trend_window:t + 1], axis=0)
        # Take a long position if the 50-days return is positive, otherwise take a short position (sign of the block return)
        signal = np.sign(block_ret)
        # Position at date t; risk adjust with volatility from previous date
        positions[t] = signal / vol
    return pl.DataFrame(positions, schema=cols).with_columns(ret[DATE]).select(DATE, *cols).fill_nan(None)

#### Optionally, the same positions can be computed in this vectorized fashion (faster to compute):
We used the explicit loop above for didactic reasons

In [ ]:
def trend_model_vectorized(ret: pl.DataFrame, trend_window: int = 50, vol_window: int = 100) -> pl.DataFrame:
    cols = [c for c in ret.columns if c != DATE]
    # calculate vol
    vol = ret.select(DATE, pl.col(cols).pow(2).rolling_sum(vol_window, min_samples=vol_window // 2).sqrt())
    # calculate signal
    signal = ret.select(DATE, pl.col(cols).rolling_sum(trend_window).sign())
    # scale with inverse vol to get pos
    return signal.select(DATE, *[(pl.col(c) / vol[c]).fill_nan(None).alias(c) for c in cols])

In [ ]:
pos = trend_model_vectorized(ret)

### Comment
The positions does not necessarily have to be binary, real valued positions are allowed. In practice the positions would be integers (number of contracts), but with alot of capital to invest, the implied rounding errors can be neglected.


## Model returns & Evaluation

If positions are indexed correctly they should now be aligned such that the position $pos_t$ only usis data up to and including $t$, be traded on day $t+1$
and generate a return between day $t+1$ and $t+2$. Since our dataframe with returns are aligned according to $r_t=p_t-p_{t-1}$, we need to shift our positions by two days to calculate the model returns at time $t$, i.e the models return from yesterday until today is: $R_t = \sum_{k=1}^M{pos^m_{t-2}r^m_t}$

In [ ]:
def model_returns(positions: pl.DataFrame, ret: pl.DataFrame) -> pl.DataFrame:
    cols = [c for c in positions.columns if c != DATE]
    gross = positions.select(DATE, *[(pl.col(c).shift(2) * ret[c]).alias(c) for c in cols])
    return gross.filter(~pl.all_horizontal(pl.col(cols).is_null())).select(
        DATE, pl.sum_horizontal(pl.col(cols)).alias("model_return")
    )


model_ret = model_returns(pos, ret)

#### Plot the cumulative returns, to see how the model is performing over time

In [ ]:
fig, ax = plt.subplots()
ax.plot(model_ret[DATE].to_numpy(), model_ret["model_return"].cum_sum().to_numpy())
ax.set_title("Cumulative returns excluding costs")
ax.grid()

#### Evaluation with transaction costs and other key figures
The trading (position changes) will induce transaction costs, epecially when trading large volumes that will have an impact on the markets. These transaction costs are often called slippage. We will provide an evaluation tool for computing returns with approximated slippage, as well as some keyfigures that can be good to take into account. See the documentation in evaluation.py for more information about the key figures.

As a general guideline to the plotted key figures, generally hedge funds tend to prefer models with high sharpe ratios that are stable over time and that are insensitive to trading costs in the form of slippage or lagged positions. Furthermore, we prefer that returns are thin-tailed (with low kurtosis), positively skewed, and that the market exposure is low (here measured by the position-bias).

In [ ]:
plot_key_figures(pos, prices)

The evaluation tool can also return a dict containing estimated key figures, either small recommended subset or a user defined set of key figures.


In [ ]:
calc_key_figures(pos, prices)

Using this function we can, for example, investigate how model performance will change with different cost assumptions:

In [ ]:
cost_coeffs = np.linspace(0, -0.1, 10)
sharpe_per_cost_coef = [
    calc_key_figures(pos, prices, costs=float(cost), key_figures=["sharpe"])["sharpe"]
    for cost in cost_coeffs
]
fig, ax = plt.subplots()
ax.plot(cost_coeffs, sharpe_per_cost_coef)
ax.set_title("Cost sensitivity")
ax.set_ylabel("Sharpe after costs")
ax.set_xlabel("Cost cofficient")
ax.grid()

## Test for forward looking bias
A useful way to test for forward looking bias, besides not lagging positions correctly, is to generate positions using a shorter history of data, and make sure that the positions are identical to those obtained when using the whole dataset.

In [ ]:
pos = trend_model_vectorized(ret)
pos_short = trend_model_vectorized(ret.head(ret.height - 20))
comparison = pos_short.join(pos, on=DATE, suffix="_full", maintain_order="left")
comparison.select(*[(pl.col(c) - pl.col(f"{c}_full")).abs().sum().alias(c) for c in instruments])

# Risk adjust our strategy

We have now defined a strategy, but we are not done yet. We need to manage the risk of the strategy. As a first approximation, we can define the standard deviation of the model returns as the risk of the strategy. We calculate this risk over a suitable horizon, for example 100 days.

In [ ]:
pos = trend_model_vectorized(ret)
model_ret = model_returns(pos, ret)
model_risk = model_ret.select(DATE, pl.col("model_return").rolling_std(100, min_samples=20).alias("risk"))
fig, ax = plt.subplots()
ax.plot(model_risk[DATE].to_numpy(), model_risk["risk"].to_numpy())
ax.set_ylabel("model risk")
ax.set_title("model risk of raw positions")
ax.grid()

As you can see, the model risk doesn't change too much over time in this example. The risk of your own strategy might vary a lot more. We want to achieve two things: 
- Scale the average strategy risk to 1.
- Reduce the fluctuations of the model risk.
- 
We can achieve both by simply scaling our strategy positions with the inverse model risk:

In [ ]:
aligned = pos.join(model_risk, on=DATE, how="left", maintain_order="left")
pos_adjusted = aligned.select(DATE, *[(pl.col(c) / pl.col("risk")).fill_nan(None).alias(c) for c in instruments])

Let's calculate model returns and model risk of this risk-adjusted strategy:


In [ ]:
model_ret = model_returns(pos_adjusted, ret)
model_risk = model_ret.select(DATE, pl.col("model_return").rolling_std(100, min_samples=20).alias("risk"))
fig, ax = plt.subplots()
ax.plot(model_risk[DATE].to_numpy(), model_risk["risk"].to_numpy())
ax.set_ylabel("model risk")
ax.set_title("model risk of adjusted positions")
ax.grid()

As you can see from the plot, the risk of the adjusted model is on average 1 and fluctuates less over time than before.

As a final step, we incorporate this risk adjustment into the strategy itself.

In [ ]:
def trend_model_vectorized_risk_adjusted(
    ret: pl.DataFrame, trend_window: int = 50, vol_window: int = 100, risk_window: int = 100
) -> pl.DataFrame:
    cols = [c for c in ret.columns if c != DATE]
    # calculate vol
    vol = ret.select(DATE, pl.col(cols).pow(2).rolling_sum(vol_window, min_samples=vol_window // 2).sqrt())
    # calculate signal
    signal = ret.select(DATE, pl.col(cols).rolling_sum(trend_window).sign())
    # scale with inverse vol to get pos
    positions = signal.select(DATE, *[(pl.col(c) / vol[c]).alias(c) for c in cols])
    # calculate model risk
    gross = positions.select(DATE, *[(pl.col(c).shift(2) * ret[c]).alias(c) for c in cols])
    risk = gross.filter(~pl.all_horizontal(pl.col(cols).is_null())).select(
        DATE, pl.sum_horizontal(pl.col(cols)).rolling_std(risk_window, min_samples=20).alias("model_risk")
    )
    # adjust with model risk
    aligned = positions.join(risk, on=DATE, how="left", maintain_order="left")
    return aligned.select(DATE, *[(pl.col(c) / pl.col("model_risk")).fill_nan(None).alias(c) for c in cols])

Finally, we run this version of the model and look at results:

In [ ]:
pos = trend_model_vectorized_risk_adjusted(ret)
plot_key_figures(pos, prices)

# Further questions:
Now that you have seen how to calculate a simple strategy, think about how it could be improved:
 - We have used certain parameter in the strategy, but they are not necessarily optimal. Can you efficiently find better parameter, for example by performing a grid search?
 - We defined the signal as $np.sign(ret.rolling(window=trend_window).sum())$, can you think of ways to improve this?
 - Can you find other strategy ideas that can be implemented in a similar way?